# Tabu Search — TSPTW-D Benchmark

TSP with Time Windows and Disruptions (TSPTW-D).  
Instances loaded from `../../datasets/tsptwd_n*.json` (generated by `datasetsgenerator.ipynb`).

| Section | Content |
|---------|----------|
| 1 | Imports & dataset loading helpers |
| 2 | Convert JSON instance → TabuSearch inputs |
| 3 | NN baseline with time-window penalty |
| 4 | Benchmark n=10, n=50, n=100 |
| 5 | Summary table + comparison chart |
| 6 | Tour visualisation with time-window feasibility |

## Section 1 — Imports & dataset loading

In [ ]:
import sys, os, time, json
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add solver directory to path
SOLVER_DIR  = os.path.dirname(os.path.abspath('__file__'))
DATASET_DIR = os.path.join(SOLVER_DIR, '..', '..', 'datasets')
sys.path.insert(0, SOLVER_DIR)

from solver import TabuSearch

# ── Tabu Search configuration ───────────────────────────────────────────────
TS_CONFIG = dict(
    tabu_tenure  = 20,
    max_iter     = 5000,
    patience     = 500,
    penalty_coeff = 1000.0,
    seed         = 42,
)

print("TSPTW-D Benchmark")
print(f"Config: {TS_CONFIG}")
print(f"Dataset dir: {os.path.realpath(DATASET_DIR)}")

## Section 2 — JSON instance → TabuSearch inputs

The datasets follow the format produced by `datasetsgenerator.ipynb`:
```json
{
  "meta": {"n_clients": int, "scale": float, "horizon": float},
  "depot":   {"id": 0, "x", "y", "a", "b", "service"},
  "clients": [{"id", "x", "y", "a", "b", "service"}, ...],
  "perturbations": [{"arc": [i,j], "t_start", "t_end", "alpha"}, ...]
}
```

In [ ]:
def load_tsptwd_instance(n_clients: int, dataset_dir: str = DATASET_DIR):
    """
    Load datasets/tsptwd_n{n_clients}.json and return tensors
    compatible with TabuSearch TSPTW-D mode.

    Returns
    -------
    coords        : (n, 2) float32 tensor  — node positions (depot first)
    time_windows  : (n, 2) float32 tensor  — [[a_i, b_i], ...]
    service_times : (n,)   float32 tensor  — s_i per node
    perturbations : list of (i, j, t_start, t_end, alpha) tuples
    meta          : dict with n_clients, scale, horizon
    """
    path = os.path.join(dataset_dir, f'tsptwd_n{n_clients}.json')
    with open(path, encoding='utf-8') as f:
        data = json.load(f)

    nodes = [data['depot']] + data['clients']   # depot at index 0
    n     = len(nodes)

    coords   = torch.tensor([[nd['x'], nd['y']] for nd in nodes], dtype=torch.float32)
    a_vals   = [nd['a'] for nd in nodes]
    b_vals   = [nd['b'] if nd['b'] is not None else float('inf') for nd in nodes]
    svc_vals = [nd['service'] for nd in nodes]

    time_windows  = torch.tensor(list(zip(a_vals, b_vals)), dtype=torch.float32)
    service_times = torch.tensor(svc_vals, dtype=torch.float32)

    perturbations = [
        (int(p['arc'][0]), int(p['arc'][1]),
         float(p['t_start']), float(p['t_end']), float(p['alpha']))
        for p in data.get('perturbations', [])
    ]

    return coords, time_windows, service_times, perturbations, data['meta']


# Quick sanity check
coords, tw, svc, perturbs, meta = load_tsptwd_instance(10)
print(f"n=10 instance loaded: {meta}")
print(f"  coords shape  : {coords.shape}")
print(f"  time_windows  : {tw.shape}")
print(f"  service_times : {svc.shape}")
print(f"  perturbations : {len(perturbs)}")

## Section 3 — NN baseline with time-window penalty

The NN baseline visits cities in greedy nearest-neighbour order **ignoring**
time windows, then evaluates the resulting schedule against TW constraints
using the same penalized objective as the TabuSearch solver.

In [ ]:
def nn_tour_from_coords(coords: torch.Tensor) -> list:
    """Greedy NN tour (ignores TW, just uses Euclidean distance)."""
    n = coords.shape[0]
    dist = torch.cdist(coords, coords)
    visited = torch.zeros(n, dtype=torch.bool)
    tour = [0]; visited[0] = True
    for _ in range(n - 1):
        d = dist[tour[-1]].clone()
        d[visited] = float('inf')
        nxt = d.argmin().item()
        tour.append(nxt); visited[nxt] = True
    return tour


def evaluate_penalized(ts_solver: TabuSearch, tour: list) -> float:
    """Evaluate the penalized TSPTW-D objective via the solver's internal method."""
    return ts_solver._penalized_obj(tour)


def gap(val, ref):
    """Gap (%) relative to reference; negative = better than ref."""
    if ref < 1e-9 or not np.isfinite(ref):
        return float('nan')
    return (val - ref) / ref * 100.0


print("Helpers defined.")

## Section 4 — Benchmark n=10, n=50, n=100

In [ ]:
BENCHMARK_SIZES = [10, 50, 100]

results  = {}   # {n: {'nn': obj, 'ts': obj, 'ts_return': return_time, 'ts_viol': violation}}
runtimes = {}   # {n: {'nn': s, 'ts': s}}
histories = {}  # {n: history list}

for n_clients in BENCHMARK_SIZES:
    print(f"\n{'='*55}")
    print(f"  n_clients = {n_clients}")
    print(f"{'='*55}")

    coords, tw, svc, perturbs, meta = load_tsptwd_instance(n_clients)
    n_nodes = coords.shape[0]   # includes depot

    # ── NN baseline ────────────────────────────────────────────────────────
    ts_eval = TabuSearch(
        coords,
        time_windows  = tw,
        service_times = svc,
        perturbations = perturbs,
        penalty_coeff = TS_CONFIG['penalty_coeff'],
    )
    t0 = time.perf_counter()
    nn_tour = nn_tour_from_coords(coords)
    nn_obj  = evaluate_penalized(ts_eval, nn_tour)
    nn_ret, nn_viol = ts_eval._compute_schedule(nn_tour)
    nn_time = time.perf_counter() - t0

    print(f"  NN  — obj={nn_obj:.2f}  return_time={nn_ret:.2f}  violation={nn_viol:.2f}  t={nn_time*1000:.1f}ms")

    # ── Tabu Search ────────────────────────────────────────────────────────
    ts = TabuSearch(
        coords,
        time_windows  = tw,
        service_times = svc,
        perturbations = perturbs,
        **TS_CONFIG,
    )
    t0 = time.perf_counter()
    best_tour, best_obj, hist = ts.run()
    ts_time = time.perf_counter() - t0

    ts_ret, ts_viol = ts._compute_schedule(best_tour)
    print(f"  TS  — obj={best_obj:.2f}  return_time={ts_ret:.2f}  violation={ts_viol:.2f}  t={ts_time:.3f}s  iters={len(hist)}")
    print(f"  Gap vs NN : {gap(best_obj, nn_obj):.1f}%")

    results[n_clients]   = {'nn_obj': nn_obj, 'ts_obj': best_obj,
                             'ts_ret': ts_ret, 'ts_viol': ts_viol,
                             'nn_ret': nn_ret, 'nn_viol': nn_viol}
    runtimes[n_clients]  = {'nn': nn_time, 'ts': ts_time}
    histories[n_clients] = hist

## Section 5 — Summary table + comparison chart

In [ ]:
# ── Summary table ───────────────────────────────────────────────────────────
print(f"{'n':>6}  {'Method':<14} {'Obj (penalized)':>16} {'Return time':>12} {'Violation':>10} {'Gap vs NN':>10} {'Time':>10}")
print('-' * 85)

for n_clients in BENCHMARK_SIZES:
    r  = results[n_clients]
    rt = runtimes[n_clients]
    g  = gap(r['ts_obj'], r['nn_obj'])
    print(f"{n_clients:>6}  {'NN':<14} {r['nn_obj']:>16.2f} {r['nn_ret']:>12.2f} {r['nn_viol']:>10.2f} {'0.0 (ref)':>10} {rt['nn']*1000:>8.1f} ms")
    print(f"{'':>6}  {'Tabu Search':<14} {r['ts_obj']:>16.2f} {r['ts_ret']:>12.2f} {r['ts_viol']:>10.2f} {g:>9.1f}% {rt['ts']:>9.3f} s")
    print()

# ── Comparison chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sizes = BENCHMARK_SIZES
nn_objs = [results[n]['nn_obj'] for n in sizes]
ts_objs = [results[n]['ts_obj'] for n in sizes]
gaps_   = [gap(results[n]['ts_obj'], results[n]['nn_obj']) for n in sizes]

# Panel 1: Objective values
ax = axes[0]
x  = np.arange(len(sizes))
w  = 0.35
ax.bar(x - w/2, nn_objs, w, label='NN',          color='#5B9BD5')
ax.bar(x + w/2, ts_objs, w, label='Tabu Search', color='#70AD47')
ax.set_xticks(x); ax.set_xticklabels([f'n={n}' for n in sizes])
ax.set_ylabel('Penalized objective'); ax.set_title('Objective comparison')
ax.legend()

# Panel 2: Gap vs NN
ax = axes[1]
bar_colors = ['#c0392b' if g > 0 else '#27ae60' for g in gaps_]
ax.bar([f'n={n}' for n in sizes], gaps_, color=bar_colors)
ax.axhline(0, color='k', linewidth=0.8)
ax.set_ylabel('Gap vs NN (%)'); ax.set_title('Gap — negative = TS beats NN')
for i, g in enumerate(gaps_):
    ax.text(i, g + (0.5 if g >= 0 else -1.5), f'{g:.1f}%', ha='center', fontsize=9)

# Panel 3: Convergence curves
ax = axes[2]
colors_hist = ['#2E86AB', '#E84855', '#A05195']
for i, n_clients in enumerate(sizes):
    hist = histories[n_clients]
    norm = hist[0] if hist[0] > 1e-9 else 1.0
    ax.plot([h / norm for h in hist], color=colors_hist[i],
            linewidth=1.5, label=f'n={n_clients}')
ax.set_xlabel('Iteration'); ax.set_ylabel('Normalised objective')
ax.set_title('Convergence'); ax.legend()

plt.suptitle('Tabu Search — TSPTW-D Benchmark', fontsize=13)
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/benchmark_tsptwd_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved figures/benchmark_tsptwd_summary.png")

## Section 6 — Tour visualisation with time-window feasibility

Each city is coloured by feasibility:
- **Green** — visited within time window
- **Red** — time-window violation

In [ ]:
def draw_tsptwd_tour(ax, coords, tour, time_windows, title,
                     service_times=None, perturbations=None,
                     penalty_coeff=1000.0, color='#2E86AB'):
    """Draw a TSPTW-D tour, colouring nodes by TW feasibility."""
    ts_eval = TabuSearch(
        coords,
        time_windows  = time_windows,
        service_times = service_times,
        perturbations = perturbations or [],
        penalty_coeff = penalty_coeff,
    )

    # Propagate schedule to get arrival times
    t = 0.0
    arrivals = []
    n = coords.shape[0]
    for k in range(n):
        city = tour[k]
        arrivals.append((city, t))
        a_i = time_windows[city, 0].item()
        b_i = time_windows[city, 1].item()
        s_i = service_times[city].item() if service_times is not None else 0.0
        depart = max(t, a_i) + s_i
        next_city = tour[(k + 1) % n]
        t = depart + ts_eval._travel_cost(city, next_city, depart)

    xy = coords.numpy()
    closed = tour + [tour[0]]
    ax.plot(xy[closed, 0], xy[closed, 1], '-', color=color, alpha=0.5, linewidth=1.2)

    for city, arr in arrivals:
        a_i = time_windows[city, 0].item()
        b_i = time_windows[city, 1].item()
        feasible = arr <= b_i
        c = '#27ae60' if feasible else '#c0392b'
        ax.plot(xy[city, 0], xy[city, 1], 'o', color=c, markersize=6, zorder=4)

    # Mark depot
    ax.plot(xy[tour[0], 0], xy[tour[0], 1], '*', color='gold', markersize=12, zorder=5)
    ax.set_title(title, fontsize=9)
    ax.set_aspect('equal'); ax.axis('off')


fig, axes = plt.subplots(2, 2, figsize=(12, 10))

pairs = [(10, 0), (10, 1), (50, 0), (100, 0)]

for ax, (n_clients, _) in zip(axes.flat, pairs):
    coords, tw, svc, perturbs, meta = load_tsptwd_instance(n_clients)
    ts = TabuSearch(coords, time_windows=tw, service_times=svc,
                    perturbations=perturbs, **TS_CONFIG)
    best_tour, best_obj, _ = ts.run()
    _, viol = ts._compute_schedule(best_tour)
    draw_tsptwd_tour(
        ax, coords, best_tour, tw,
        title=f'TS  n={n_clients}  obj={best_obj:.1f}  viol={viol:.1f}',
        service_times=svc, perturbations=perturbs,
    )

green_p = mpatches.Patch(color='#27ae60', label='TW satisfied')
red_p   = mpatches.Patch(color='#c0392b', label='TW violated')
gold_p  = mpatches.Patch(color='gold',    label='Depot')
fig.legend(handles=[green_p, red_p, gold_p], loc='lower center', ncol=3, fontsize=10)
plt.suptitle('TSPTW-D tours — Tabu Search (green=TW ok, red=TW violated)', fontsize=12)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('figures/benchmark_tsptwd_tours.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved figures/benchmark_tsptwd_tours.png")